In [ ]:
# Built-in modules

# Core scientific stack

# Setup autoreload
%load_ext autoreload
%autoreload 2

# Local module imports

# Jupyter settings
%matplotlib inline

In [ ]:
df_path = Path(input('Please enter the full path for the dataframe:\n'))

In [ ]:
df = pd.read_csv(df_path)
df.head()


In [ ]:
# Filter out missing data/inconsistent timepoints
df.loc[df['elapsed time (hr)']== -0.5, 'elapsed time (hr)'] = -1
df_filt = rd.filter_incomplete_data(df, 'elapsed time (hr)', max_num_incomplete=2)

# Compute changes in compaction values between timepoints
base_cmp_cols = ['cell area', 'CAAX-positive area', 'compacted area', '% compaction']
df_filt, cmp_cols = rd.compute_change_cols(df_filt, base_cmp_cols)
                                        
# Average compaction-related values by biorep
groupbycols = ['tx', 'experiment', 'elapsed time (hr)']
biorep_cmp_df = rd.compute_means_by_biorep(df_filt, groupbycols, cmp_cols, omit_col='omit')
biorep_cmp_df_path = df_path.parent / 'biorep_cmp_data.csv'
utils.safe_save_csv(biorep_cmp_df, biorep_cmp_df_path)


# Make graphs

In [ ]:
graphs_dirpath = Path(input())

In [ ]:
stats_combined = pd.DataFrame()
stats_df_path = df_path.parent / 'stats_df.csv'

## Control actin plots

In [ ]:
# --- Filter for control actin intensity columns ---
time_col = 'elapsed time (hr)'
ctrl_actin_df = df_filt[df_filt['tx']=='DMSO']
ctrl_actin_df = rd.select_timepoints(ctrl_actin_df, time_col, 0, 10)

# --- Normalize actin intensity columns ---

# get initial actin intensity of each cell (will normalize by this)
norm_ref = ctrl_actin_df.groupby('UID')['mean actin int (cell)'].transform('first')

# normalize all of the actin columns by the reference values
actin_cols = [
    col for col in ctrl_actin_df.columns 
    if 'actin int' in col and 'leading up to cmp' not in col
]

df_norm = ctrl_actin_df[actin_cols].div(norm_ref, axis=0) * 100
df_norm = df_norm.add_prefix('norm ')
norm_actin_cols = df_norm.columns.tolist()
ycols = cmp_cols + actin_cols + norm_actin_cols
ctrl_actin_df = pd.concat([ctrl_actin_df, df_norm], axis=1)

# --- Compute biological replicate means for actin intensity ---
biorep_actin_df = rd.compute_means_by_biorep(ctrl_actin_df, groupbycols, ycols, omit_col='actin omit')
biorep_actin_df_path = df_path.parent / 'biorep_actin_data.csv'
utils.safe_save_csv(biorep_actin_df, biorep_actin_df_path)


In [ ]:
time_col = 'elapsed time (hr)'

ctrl_biorep_actin_df = biorep_actin_df[biorep_actin_df['tx']=='DMSO']
ctrl_biorep_actin_df = rd.select_timepoints(ctrl_biorep_actin_df, time_col, 0, 10)

# get initial actin intensity of each cell (will normalize by this)
norm_ref = df_filt.groupby('UID')['mean actin int (cell)'].transform('first')

prefix = 'ctrl_actinint'
data = ctrl_biorep_actin_df
ycol = 'mean actin int (a.u.) / cell'
subject = 'experiment'
group_category = 'region'

norm_actin_cols = [col for col in data.columns if 'norm mean' in col]

paired_actin_cols = [('norm mean actin int (caax)', 'norm mean actin int (compacted)'), ('norm mean actin int in uncompacted regions that stay uncompacted next frame', 'norm mean actin int in uncompacted regions that will compact next frame')]
paired_actin_labels = [('non-compact', 'compact'), ('stays non-compact', 'compacts next frame')]


for ycols, labels in zip(paired_actin_cols, paired_actin_labels):

    comparison_label = {
        'columns': ycols,
        'labels': labels,
        'name': f'{prefix}_{ptd.clean_column_name(labels[0])}_vs_{ptd.clean_column_name(labels[1])}'
    }
    
    # Re-arrange dataframe and run stats
    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=True
    )


    
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    labels = [ptd.wrap_text(label) for label in labels]
    
    # plot timelapse graphs
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=ycols,
        palette=ps.cmpreg_palette,
        group_labels=labels,
        graphname=comparison_label['name'],
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=ps.timelapse_figsize
    )

    
    # Plot scatter plots for first and last timepoints
    timepoints = sorted(df_long[time_col].dropna().unique())
    tp_list = [timepoints[0], timepoints[-1]]
    
    
    for tp in tp_list:
        ptd.plot_individual_tp(
            df_long=df_long,
            stats_df=stats_df,
            tp=tp,
            xcol=time_col,
            ycol=ycol,
            group_col=group_category,
            comparison_label=comparison_label['name'],
            labels=labels,
            palette=ps.cmpreg_palette,
            savepath=graphs_dirpath / f"{comparison_label['name']}_tp{tp:.0f}.svg",
            figsize=ps.scatter_figsize
        )
        
    utils.safe_save_csv(stats_combined, stats_df_path)


## Control compaction plots

In [ ]:
ctrl_biorep_cmp_df = biorep_actin_df[biorep_cmp_df['tx']=='DMSO']
ctrl_biorep_cmp_df = rd.select_timepoints(ctrl_biorep_cmp_df, time_col, 0, 10)

prefix = 'ctrl_cmp'
data = ctrl_biorep_cmp_df
time_col = 'elapsed time (hr)'
subject = 'experiment'
group_category = 'tx'
hue_order = ['DMSO']
palette = [ps.ctrl_color]

# Plot timelapse graphs
for ycol in cmp_cols:
    
    ptd.plot_timelapse_lines(
        df_long=data,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=palette,
        graphname=f'{prefix}_{ptd.clean_column_name(ycol)}',
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=ps.timelapse_figsize
    )

ctrl_biorep_cmp_df_path = df_path.parent / 'ctrl_biorep_cmp_data.csv'
utils.safe_save_csv(ctrl_biorep_cmp_df, ctrl_biorep_cmp_df_path)

# Control vs latrunculin plots

In [ ]:
ctrlvslat_biorep_cmp_df = biorep_cmp_df[biorep_cmp_df['tx'].isin(['DMSO', 'latA'])]

prefix = 'ctrlvslat_cmp'
data = ctrlvslat_biorep_cmp_df
time_col = 'elapsed time (hr)'
subject = 'experiment'
group_category = 'tx'
hue_order = ['DMSO', 'latA']
palette = ps.ctrllat_palette

for ycol in cmp_cols:
    comparison_label = {
        'columns': ycol,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}'
    }
    
    # Run stats
    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False
    )

    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # plot timelapse graphs
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=palette,
        tx_line=0,
        graphname=comparison_label['name'],
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=ps.timelapse_figsize
    )
        
    utils.safe_save_csv(stats_combined, stats_df_path)


In [ ]:
# # --- Compute biological replicate means for actin int leading up to compaction ---

# Create a copy of the dataframe with values for actin leading up to compaction
actin_before_cmp_df = df[df['tx']=='DMSO'].copy()
time_col = 'time relative to compaction (hr)'
value_col = 'mean actin int leading up to cmp'
actin_before_cmp_df = rd.filter_incomplete_data(actin_before_cmp_df, time_col, value_col=value_col, max_num_incomplete=2)


# Average compaction-related values by biorep
groupbycols = ['tx', 'experiment', time_col]
actinbeforecmp_cols = [value_col, 'mean actin int in regions that remain CAAX+']
ctrl_biorep_actinbeforecmp_df = rd.compute_means_by_biorep(actin_before_cmp_df, groupbycols, actinbeforecmp_cols, omit_col='actin omit')
ctrl_biorep_actinbeforecmp_df_path = df_path.parent / 'ctrl_biorep_actinbeforecmp_data.csv'
utils.safe_save_csv(ctrl_biorep_actinbeforecmp_df, ctrl_biorep_actinbeforecmp_df_path)

ctrl_biorep_actinbeforecmp_df.head()

In [ ]:
df_subset = ctrl_biorep_actinbeforecmp_df

fig, ax = plt.subplots(figsize=timelapse_figsize)
sns.lineplot(x='time relative to compaction (hr)', y='mean actin int leading up to cmp', data=df_subset, label='compacts at time 0', errorbar='se', color=cmp_color)
#sns.lineplot(x='time relative to compaction (hr)', y='mean actin int in regions that remain CAAX+', data=df_subset, label='stays non-compact', errorbar='se', color=caax_color)
ptd.style_standard_plot(ax)
plt.ylabel('mean LifeAct intensity (au) per cell')
plt.savefig(graphs_dirpath/'lifeAct_int_prior_to_cmp.png')
plt.show()

#repeated_measures_with_posthoc(dfm, subject_col = 'experiment', group_col = 'region', time_col='time relative to compaction (hr)', value_col='value', alpha=0.05)


In [ ]:
cmp_zone_df_path = Path(input())

In [ ]:
cmp_zone_df = pd.read_csv(cmp_zone_df_path)
cmp_zone_df.head()

In [ ]:
# filter compaction zone df
time_col = 'elapsed time (hr)'

cmp_zone_df.loc[cmp_zone_df[time_col]== -0.5, time_col] = -1
cmp_zone_df_filt = rd.filter_incomplete_data(cmp_zone_df, time_col, max_num_incomplete=2)
cmp_zone_df_filt = rd.select_timepoints(cmp_zone_df_filt, time_col, -1, 10)
cmp_zone_df_filt.head()

In [ ]:
#------Calculate mean number and area of compaction zones by cell------
groupbycols = ['tx', 'experiment', time_col]
sum_cmp_groupby = groupbycols + ['UID']

cmp_area_col = 'area (microns^2)'
sum_cmp_zone_df = cmp_zone_df_filt.groupby(sum_cmp_groupby, as_index=False).agg({
    cmp_area_col: ['mean', 'count']
})

# Flatten MultiIndex columns
sum_cmp_zone_df.columns = ['_'.join(col).strip() if col[1] else col[0] for col in sum_cmp_zone_df.columns.values]

# Rename columns
cmp_zone_ycols = ['mean cmp zone area (μm²)', 'num cmp zones']
sum_cmp_zone_df = sum_cmp_zone_df.rename(columns={
    f'{cmp_area_col}_mean': cmp_zone_ycols[0],
    f'{cmp_area_col}_count': cmp_zone_ycols[1]
})

# Compute changes in compaction values between timepoints
sum_cmp_zone_df, cmp_zone_ycols = rd.compute_change_cols(sum_cmp_zone_df, cmp_zone_ycols, time_col=time_col)

sum_cmp_zone_df.head()

In [ ]:
#------Calculate averages by biological replicates------
biorep_sum_cmp_zone_df = rd.compute_means_by_biorep(sum_cmp_zone_df, groupbycols, cmp_zone_ycols)
biorep_sum_cmp_zone_df.to_csv(df_path.parent/'biorep_sum_cmp_zone_df.csv')
biorep_sum_cmp_zone_df.head()

In [ ]:
prefix = 'ctrlvslatA'
data = biorep_sum_cmp_zone_df[biorep_sum_cmp_zone_df['tx'].isin(['DMSO', 'latA'])]
xcol = 'elapsed time (hr)'

time_col = 'elapsed time (hr)'
cmp_area_col = 'area (microns^2)'

cum_change_cmp_cols = [col for col in cmp_zone_ycols if 'cumulative change' in col]
hue_order = ['DMSO', 'latA']
subject = 'experiment'
time = time_col
group_category = 'tx'
value = ycol


for ycol in cum_change_cmp_cols:

    comparison_label = {
        'columns': ycols,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}'
    }

    # Calculate stats
    df_long, stats_df = rs.run_repeated_measures_stats(data, subject, time, group_category, ycol, comparison_label, melt_df=False)
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # Plot timelapse data
    ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=ps.ctrllat_palette,
        group_labels=None,
        graphname=comparison_label['name'],
        tx_line=0,
        save_dir=graphs_dirpath,
        stats_df=stats_df, 
        figsize=ps.timelapse_figsize
    )
